# Finance Sentiment Analysis

## Step 1 — Select the use case

**Use case:** Finance sentiment analysis.

**Goal:** assess whether financial news and social-media-style posts express positive or negative sentiment about financial assets and markets.

In [9]:
# Run once in Jupyter if these packages are not already installed.
%pip install faiss-cpu numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 5.3 MB/s  0:00:015.9 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 6.6 MB/s  0:00:00m 6.4 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [faiss-cpu]━ 1/2 [faiss-cpu]
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import csv
import json
import math
import re
from collections import Counter

import faiss
import numpy as np

WORK_DIR = Path.cwd() / 'finance_workshop_data'
WORK_DIR.mkdir(exist_ok=True)
CSV_FILE = WORK_DIR / 'finance_sentiment_data.csv'
EMBEDDINGS_FILE = WORK_DIR / 'finance_embeddings.json'
FAISS_INDEX_FILE = WORK_DIR / 'finance_sentiment.index'

## Step 2 — Create and label the data

Each row is a text chunk with a manually assigned sentiment label.

In [3]:
records = [
    {'text': 'Federal Reserve hints at potential interest rate cuts by Q4, sparking market rally.', 'label': 'positive'},
    {'text': 'Tech giant reports quarterly earnings miss, shares plummet 5% in after-hours trading.', 'label': 'negative'},
    {'text': 'New regulatory framework for crypto assets announced, bringing clarity to institutional investors.', 'label': 'positive'},
    {'text': 'Retail sales figures come in lower than expected, raising concerns about consumer spending.', 'label': 'negative'},
    {'text': 'Global oil prices stabilize as supply chain disruptions ease in Southeast Asia.', 'label': 'positive'},
    {'text': 'Major bank faces investigation over alleged money laundering failures.', 'label': 'negative'},
    {'text': 'AI-driven productivity gains expected to boost corporate margins across the S&P 500.', 'label': 'positive'},
    {'text': 'Inflation remains stubbornly high, putting pressure on central banks to maintain tight policy.', 'label': 'negative'},
    {'text': 'Emerging markets show resilience despite global economic headwinds.', 'label': 'positive'},
    {'text': 'Cybersecurity breach at leading insurance firm exposes millions of customer records.', 'label': 'negative'},
    {'text': 'Green energy stocks surge as government unveils massive subsidies for hydrogen power.', 'label': 'positive'},
    {'text': 'Corporate debt levels reach historic highs, increasing default risks in a high-rate environment.', 'label': 'negative'},
    {'text': 'Consumer confidence index hits two-year high as employment numbers stay strong.', 'label': 'positive'},
    {'text': 'Trade tensions between major economies threaten to disrupt semiconductor supply chains.', 'label': 'negative'},
    {'text': 'Luxury goods sector reports record growth driven by demand in Asian markets.', 'label': 'positive'},
]

with CSV_FILE.open('w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=['text', 'label'])
    writer.writeheader()
    writer.writerows(records)

print(f'Saved {len(records)} labeled records to {CSV_FILE}')
records[:3]

Saved 15 labeled records to /Users/amirsayed/Documents/Codex/2026-09-15/le/outputs/finance_workshop_data/finance_sentiment_data.csv


[{'text': 'Federal Reserve hints at potential interest rate cuts by Q4, sparking market rally.',
  'label': 'positive'},
 {'text': 'Tech giant reports quarterly earnings miss, shares plummet 5% in after-hours trading.',
  'label': 'negative'},
 {'text': 'New regulatory framework for crypto assets announced, bringing clarity to institutional investors.',
  'label': 'positive'}]

## Step 3 — Convert chunks into embeddings and store them in FAISS

TF-IDF provides the baseline text embeddings. Vectors are L2-normalized, so FAISS `IndexFlatIP` returns cosine-similarity scores.

In [5]:
from pathlib import Path
import csv
import json
import math
import re
from collections import Counter

import faiss
import numpy as np

WORK_DIR = Path.cwd() / 'finance_workshop_data'
WORK_DIR.mkdir(exist_ok=True)
CSV_FILE = WORK_DIR / 'finance_sentiment_data.csv'
EMBEDDINGS_FILE = WORK_DIR / 'finance_embeddings.json'
FAISS_INDEX_FILE = WORK_DIR / 'finance_sentiment.index'


records = [
    {'text': 'Federal Reserve hints at potential interest rate cuts by Q4, sparking market rally.', 'label': 'positive'},
    {'text': 'Tech giant reports quarterly earnings miss, shares plummet 5% in after-hours trading.', 'label': 'negative'},
    {'text': 'New regulatory framework for crypto assets announced, bringing clarity to institutional investors.', 'label': 'positive'},
    {'text': 'Retail sales figures come in lower than expected, raising concerns about consumer spending.', 'label': 'negative'},
    {'text': 'Global oil prices stabilize as supply chain disruptions ease in Southeast Asia.', 'label': 'positive'},
    {'text': 'Major bank faces investigation over alleged money laundering failures.', 'label': 'negative'},
    {'text': 'AI-driven productivity gains expected to boost corporate margins across the S&P 500.', 'label': 'positive'},
    {'text': 'Inflation remains stubbornly high, putting pressure on central banks to maintain tight policy.', 'label': 'negative'},
    {'text': 'Emerging markets show resilience despite global economic headwinds.', 'label': 'positive'},
    {'text': 'Cybersecurity breach at leading insurance firm exposes millions of customer records.', 'label': 'negative'},
    {'text': 'Green energy stocks surge as government unveils massive subsidies for hydrogen power.', 'label': 'positive'},
    {'text': 'Corporate debt levels reach historic highs, increasing default risks in a high-rate environment.', 'label': 'negative'},
    {'text': 'Consumer confidence index hits two-year high as employment numbers stay strong.', 'label': 'positive'},
    {'text': 'Trade tensions between major economies threaten to disrupt semiconductor supply chains.', 'label': 'negative'},
    {'text': 'Luxury goods sector reports record growth driven by demand in Asian markets.', 'label': 'positive'},
]

with CSV_FILE.open('w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=['text', 'label'])
    writer.writeheader()
    writer.writerows(records)

print(f'Saved {len(records)} labeled records to {CSV_FILE}')
records[:3]

STOP_WORDS = {'a', 'an', 'and', 'as', 'at', 'by', 'for', 'from', 'in', 'is', 'it', 'of', 'on', 'or', 'the', 'to', 'with', 'across', 'about', 'after', 'against', 'into', 'over', 'than', 'that', 'this', 'their'}

def tokenize(text):
    return [word for word in re.findall(r'[a-z0-9]+', text.lower())
            if word not in STOP_WORDS and len(word) > 1]

def l2_normalize(vector):
    magnitude = math.sqrt(sum(value * value for value in vector))
    return vector if magnitude == 0 else [value / magnitude for value in vector]

tokenized_documents = [tokenize(record['text']) for record in records]
document_frequency = Counter(token for tokens in tokenized_documents for token in set(tokens))
vocabulary = sorted(document_frequency)
word_to_index = {word: index for index, word in enumerate(vocabulary)}

embedded_records = []
for record, tokens in zip(records, tokenized_documents):
    term_frequency = Counter(tokens)
    vector = [0.0] * len(vocabulary)
    for word, count in term_frequency.items():
        idf = math.log((1 + len(records)) / (1 + document_frequency[word])) + 1
        vector[word_to_index[word]] = (count / len(tokens)) * idf
    embedded_records.append({**record, 'embedding': l2_normalize(vector)})

vectors = np.asarray([record['embedding'] for record in embedded_records], dtype=np.float32)
index = faiss.IndexFlatIP(len(vocabulary))  # inner product = cosine similarity for normalized vectors
index.add(vectors)
faiss.write_index(index, str(FAISS_INDEX_FILE))

with EMBEDDINGS_FILE.open('w', encoding='utf-8') as file:
    json.dump({'vocabulary': vocabulary, 'records': embedded_records}, file, indent=2)

print(f'Created {index.ntotal} embeddings with {index.d} dimensions.')
print(f'FAISS index saved to: {FAISS_INDEX_FILE}')

# Verification: reopen the stored FAISS index.
saved_index = faiss.read_index(str(FAISS_INDEX_FILE))
print(f'Verified index: {saved_index.ntotal} vectors, {saved_index.d} dimensions')





Saved 15 labeled records to /Users/amirsayed/Documents/Codex/2026-09-15/le/outputs/finance_workshop_data/finance_sentiment_data.csv
Created 15 embeddings with 137 dimensions.
FAISS index saved to: /Users/amirsayed/Documents/Codex/2026-09-15/le/outputs/finance_workshop_data/finance_sentiment.index
Verified index: 15 vectors, 137 dimensions
Question: How are inflation and interest rates affecting financial markets?
Top matching records:
1. score=0.1851 | label=negative | Inflation remains stubbornly high, putting pressure on central banks to maintain tight policy.
2. score=0.1838 | label=positive | Federal Reserve hints at potential interest rate cuts by Q4, sparking market rally.
3. score=0.1658 | label=positive | Emerging markets show resilience despite global economic headwinds.
4. score=0.1493 | label=positive | Luxury goods sector reports record growth driven by demand in Asian markets.
5. score=0.0000 | label=positive | Global oil prices stabilize as supply chain disruptions ease i

## Step 4 — Create the question

Test the system with a real finance question and check whether the nearest matches are mostly negative, which is what we would expect for a question about inflation and interest rates hurting markets.

In [ ]:
question = 'Is AI boom good for the market?'

question_tokens = [
    word for word in re.findall(r'[a-z0-9]+', question.lower())
    if word not in STOP_WORDS and len(word) > 1
]
question_frequency = Counter(question_tokens)

question_vector = [0.0] * len(vocabulary)
for word, count in question_frequency.items():
    if word in word_to_index:
        idf = math.log((1 + len(records)) / (1 + document_frequency[word])) + 1
        question_vector[word_to_index[word]] = (count / len(question_tokens)) * idf

question_vector = np.asarray([l2_normalize(question_vector)], dtype=np.float32)

scores, indices = index.search(question_vector, 10)
print(f'Question: {question}')
print('Top matching records:')
for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    match = records[int(idx)]
    print(f'{rank}. score={score:.4f} | label={match["label"]} | {match["text"]}')

nearest_labels = [records[int(idx)]['label'] for idx in indices[0]]
label_counts = Counter(nearest_labels)
print('Nearest labels:', label_counts)
print('Most common label among matches:', label_counts.most_common(1)[0][0])

if label_counts.most_common(1)[0][0] == 'negative':
    print('✅ Sentiment check passed: the query is matching negative market sentiment as expected.')
elif label_counts.most_common(1)[0][0] == 'positive':
    print('✅ Sentiment check passed: the query is matching positive market sentiment as expected.')
else:
    print('⚠️ Sentiment check looks off: the question is not matching the expected negative signal.')


Question: Is AI boom good for the market?
Top matching records:
1. score=0.2460 | label=positive | AI-driven productivity gains expected to boost corporate margins across the S&P 500.
2. score=0.2156 | label=positive | Federal Reserve hints at potential interest rate cuts by Q4, sparking market rally.
3. score=0.0000 | label=negative | Cybersecurity breach at leading insurance firm exposes millions of customer records.
4. score=0.0000 | label=positive | Emerging markets show resilience despite global economic headwinds.
5. score=0.0000 | label=negative | Inflation remains stubbornly high, putting pressure on central banks to maintain tight policy.
6. score=0.0000 | label=negative | Major bank faces investigation over alleged money laundering failures.
7. score=0.0000 | label=positive | Global oil prices stabilize as supply chain disruptions ease in Southeast Asia.
8. score=0.0000 | label=negative | Retail sales figures come in lower than expected, raising concerns about consumer spendi

## Step 5 — Return the k nearest neighbors using cosine similarity

Because the vectors are L2-normalized, the FAISS dot product is equivalent to cosine similarity. This step returns the nearest records for a question using the saved index.

In [24]:
k = 5
question = 'Are inflation and interest rates affecting financial markets and stock prices good or bad?'

question_tokens = [
    word for word in re.findall(r'[a-z0-9]+', question.lower())
    if word not in STOP_WORDS and len(word) > 1
]
question_frequency = Counter(question_tokens)

question_vector = [0.0] * len(vocabulary)
for word, count in question_frequency.items():
    if word in word_to_index:
        idf = math.log((1 + len(records)) / (1 + document_frequency[word])) + 1
        question_vector[word_to_index[word]] = (count / len(question_tokens)) * idf

query_vector = np.asarray([l2_normalize(question_vector)], dtype=np.float32)
similarity_scores, nearest_indices = index.search(query_vector, k)

print(f'Question: {question}')
print(f'k={k} nearest neighbors by cosine similarity:')
for rank, (score, idx) in enumerate(zip(similarity_scores[0], nearest_indices[0]), start=1):
    match = records[int(idx)]
    print(f'{rank}. cosine_similarity={score:.4f} | label={match["label"]} | text={match["text"]}')

Question: Are inflation and interest rates affecting financial markets and stock prices good or bad?
k=5 nearest neighbors by cosine similarity:
1. cosine_similarity=0.1674 | label=positive | text=Global oil prices stabilize as supply chain disruptions ease in Southeast Asia.
2. cosine_similarity=0.1585 | label=negative | text=Inflation remains stubbornly high, putting pressure on central banks to maintain tight policy.
3. cosine_similarity=0.1574 | label=positive | text=Federal Reserve hints at potential interest rate cuts by Q4, sparking market rally.
4. cosine_similarity=0.1420 | label=positive | text=Emerging markets show resilience despite global economic headwinds.
5. cosine_similarity=0.1279 | label=positive | text=Luxury goods sector reports record growth driven by demand in Asian markets.


## Step 6 — Convert similarity score to percentage

Each retrieved result from FAISS comes with a cosine similarity score in the range $[-1, 1]$. To make it easier to interpret, convert each score to a percentage by scaling it to $[0, 100]$.

In [25]:
# Convert cosine similarity to percentage for each returned result
similarity_pct = [(score + 1) / 2 * 100 for score in similarity_scores[0]]

print(f'Question: {question}')
print('Similarity percentage for each top match:')
for rank, (score, pct, idx) in enumerate(zip(similarity_scores[0], similarity_pct, nearest_indices[0]), start=1):
    match = records[int(idx)]
    print(f'{rank}. cosine_similarity={score:.4f} | similarity_pct={pct:.2f}% | label={match["label"]} | text={match["text"]}')

Question: Are inflation and interest rates affecting financial markets and stock prices good or bad?
Similarity percentage for each top match:
1. cosine_similarity=0.1674 | similarity_pct=58.37% | label=positive | text=Global oil prices stabilize as supply chain disruptions ease in Southeast Asia.
2. cosine_similarity=0.1585 | similarity_pct=57.93% | label=negative | text=Inflation remains stubbornly high, putting pressure on central banks to maintain tight policy.
3. cosine_similarity=0.1574 | similarity_pct=57.87% | label=positive | text=Federal Reserve hints at potential interest rate cuts by Q4, sparking market rally.
4. cosine_similarity=0.1420 | similarity_pct=57.10% | label=positive | text=Emerging markets show resilience despite global economic headwinds.
5. cosine_similarity=0.1279 | similarity_pct=56.39% | label=positive | text=Luxury goods sector reports record growth driven by demand in Asian markets.
